# Xay Dung & Chuan Hoa Bang Lich Su Bien Dong: job_snapshots

Notebook nay tap trung xay dung bang lich su trang thai theo thoi gian **`job_snapshots`** trong kien truc Silver Layer:
- Ap dung mo hinh Daily Periodic Snapshot / SCD Type 2 de luu giu trang thai cua tin tuyen dung tai tung moc thoi gian.
- Phuc vu cac bai toan phan tich time-series chuyen sau: bien dong khung luong (salary trend), thoi gian ton tai trung binh cua tin tuyen dung (time-to-fill / job lifespan), va ty le tin hoat dong theo thoi gian thuc.
- Kiem tra nghiem ngat rang buoc DDL PostgreSQL: khoa chinh `snapshot_id`, khoa ngoai tham chieu `jobs(job_id)`, va rang buoc duy nhat `UNIQUE(job_id, snapshot_date)`.

In [1]:
import os
import re
import json
from collections import Counter
from datetime import datetime, date
import pandas as pd
import numpy as np

# Cau hinh hien thi pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Moi truong xu ly bang job_snapshots da san sang!')

Moi truong xu ly bang job_snapshots da san sang!


## 1. Nap Du Lieu Bronze va Dong Bo Voi Bang jobs

Dong bo thong tin muc luong va thoi gian thu thap tu bang `jobs` (`02_jobs.ipynb`).

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
jobs_nb_path = os.path.abspath('02_jobs.ipynb')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

df_listings = pd.DataFrame(bronze_listings)
df_details = pd.DataFrame(bronze_details)

# Inner join theo dung thu tu dong bo nhu 02_jobs.ipynb
df_merged = pd.merge(
    df_listings,
    df_details,
    on=['source', 'source_job_id'],
    how='inner',
    suffixes=('_list', '_detail')
)

# Dong bo job_id (1-indexed)
df_merged['job_id'] = df_merged.index + 1

# Tuan thu bo parser luong tu 02_jobs.ipynb
with open(jobs_nb_path, 'r', encoding='utf-8') as f:
    nb2 = json.load(f)
import re
scope2 = {'pd': pd, 're': re}
exec(''.join(nb2['cells'][7]['source']), scope2)
parse_salary = scope2['parse_salary']

sal_records = [parse_salary(s) for s in df_merged['salary_raw_list']]
df_sal = pd.DataFrame(sal_records)
df_merged['salary_min_vnd'] = df_sal['salary_min_vnd']
df_merged['salary_max_vnd'] = df_sal['salary_max_vnd']

print(f'Tong so tin tuyen dung dong bo: {len(df_merged):,} tin')
print(f'So tin co thong so luong cu the: {df_merged["salary_max_vnd"].notna().sum():,} tin')
print(f'Phan bo ngay crawl tren tap du lieu: {Counter(df_merged["raw_date_list"])}')

🧪 Kiểm tra thử nghiệm parser lương:
  • "25 - 40 triệu" -> Min: 25000000.0 | Max: 40000000.0 | Type: range | Curr: VND
  • "$ 1,000-2,500 /tháng" -> Min: 25400000.0 | Max: 63500000.0 | Type: range | Curr: USD
  • "Tới 30 triệu" -> Min: None | Max: 30000000.0 | Type: max_only | Curr: VND
  • "15000000" -> Min: 15000000.0 | Max: 15000000.0 | Type: fixed | Curr: VND
  • "14000000 - 18000000 MONTH" -> Min: 14000000.0 | Max: 18000000.0 | Type: range | Curr: VND
  • "$ 400-1,500 /tháng" -> Min: 10160000.0 | Max: 38100000.0 | Type: range | Curr: USD
  • "Thương lượng" -> Min: None | Max: None | Type: negotiable | Curr: VND
Tong so tin tuyen dung dong bo: 2,327 tin
So tin co thong so luong cu the: 982 tin
Phan bo ngay crawl tren tap du lieu: Counter({'2026-10-11': 1510, '2026-10-12': 817})


## 2. Thiet Ke Pipeline Tao Snapshot Theo Ngay (Periodic Snapshot Pipeline)

Tao ban ghi snapshot theo chu ky hang ngay:
- Ngay 2026-10-11: Chup trang thai dot crawl 1 (1,510 tin).
- Ngay 2026-10-12: Chup trang thai toan bo tin tiep tuc ton tai tren he thong va tin moi xuat hien (2,327 tin).
- Xac dinh trang thai `is_active` dua tren han nop ho so (`deadline_raw`).

In [3]:
snapshots = []
snapshot_id_counter = 1
now_iso = datetime.now().isoformat()

# 1. Snapshot Dot 1: Ngay 2026-10-11 (Cac tin thu thap vao ngay 2026-10-11)
df_day1 = df_merged[df_merged['raw_date_list'] == '2026-10-11']
for idx, row in df_day1.iterrows():
    job_id = row['job_id']
    snap_date = '2026-10-11'
    dl_str = str(row.get('deadline_raw', '') or '')[:10]
    is_act = True if (not dl_str or dl_str >= snap_date) else False
    
    snapshots.append({
        'snapshot_id': snapshot_id_counter,
        'job_id': job_id,
        'snapshot_date': snap_date,
        'salary_min_vnd': row['salary_min_vnd'],
        'salary_max_vnd': row['salary_max_vnd'],
        'is_active': is_act,
        'created_at': now_iso
    })
    snapshot_id_counter += 1

# 2. Snapshot Dot 2: Ngay 2026-10-12 (Toan bo 2,327 tin tai ngay thu 2)
for idx, row in df_merged.iterrows():
    job_id = row['job_id']
    snap_date = '2026-10-12'
    dl_str = str(row.get('deadline_raw', '') or '')[:10]
    is_act = True if (not dl_str or dl_str >= snap_date) else False
    
    snapshots.append({
        'snapshot_id': snapshot_id_counter,
        'job_id': job_id,
        'snapshot_date': snap_date,
        'salary_min_vnd': row['salary_min_vnd'],
        'salary_max_vnd': row['salary_max_vnd'],
        'is_active': is_act,
        'created_at': now_iso
    })
    snapshot_id_counter += 1

df_job_snapshots = pd.DataFrame(snapshots)

print('Khoi tao bang job_snapshots thanh cong:')
print(f'  - Tong so ban ghi snapshot: {len(df_job_snapshots):,}')
print(f'  - Tinh duy nhat cua Primary Key snapshot_id: {df_job_snapshots["snapshot_id"].is_unique}')
print(f'  - Tinh duy nhat cua Unique Constraint (job_id, snapshot_date): {len(df_job_snapshots) == len(df_job_snapshots.drop_duplicates(subset=["job_id", "snapshot_date"]))}')
print(f'  - Toan ven khoa ngoai job_id: {df_job_snapshots["job_id"].between(1, len(df_merged)).all()} (100.0% hop le)')
print(f'  - Ty le tin hoat dong (is_active = True): {(df_job_snapshots["is_active"] == True).sum():,} / {len(df_job_snapshots)} ({(df_job_snapshots["is_active"] == True).mean()*100:.1f}%)')

Khoi tao bang job_snapshots thanh cong:
  - Tong so ban ghi snapshot: 3,837
  - Tinh duy nhat cua Primary Key snapshot_id: True
  - Tinh duy nhat cua Unique Constraint (job_id, snapshot_date): True
  - Toan ven khoa ngoai job_id: True (100.0% hop le)
  - Ty le tin hoat dong (is_active = True): 3,809 / 3837 (99.3%)


## 3. Danh Gia Tinh Toan Ven Rang Buoc & Chat Luong Du Lieu Time-Series

In [4]:
# 1. Kiem tra tinh logic cua muc luong: min <= max
valid_sal = df_job_snapshots[df_job_snapshots['salary_min_vnd'].notna() & df_job_snapshots['salary_max_vnd'].notna()]
is_sal_logical = (valid_sal['salary_min_vnd'] <= valid_sal['salary_max_vnd']).all()

# 2. Kiem tra format ngay snapshot
is_date_valid = df_job_snapshots['snapshot_date'].str.match(r'^\d{4}-\d{2}-\d{2}$').all()

print('--- KIEM TRA LOGIC DU LIEU BANG job_snapshots ---')
print(f'Logic luong toi thieu <= luong toi da: {is_sal_logical} (Kiem tra tren {len(valid_sal):,} ban ghi co du 2 dau luong)')
print(f'Format ngay snapshot chuan ISO YYYY-MM-DD: {is_date_valid}')

--- KIEM TRA LOGIC DU LIEU BANG job_snapshots ---
Logic luong toi thieu <= luong toi da: True (Kiem tra tren 1,326 ban ghi co du 2 dau luong)
Format ngay snapshot chuan ISO YYYY-MM-DD: True


## 4. Phan Tich Thong Ke Bien Dong Theo Thoi Gian & Preview Bang job_snapshots

In [5]:
print('--- 1. CAU TRUC BANG job_snapshots (INFO) ---')
print(df_job_snapshots.info())

print('\n--- 2. PHAN BO BAN GHI SNAPSHOT THEO NGAY ---')
date_counts = df_job_snapshots['snapshot_date'].value_counts().sort_index()
df_date_summary = pd.DataFrame({'So snapshot': date_counts, 'Ty le (%)': (date_counts / len(df_job_snapshots) * 100).round(2)})
print(df_date_summary.to_string())

print('\n--- 3. THONG KE MUC LUONG TRUNG BINH THEO TUNG NGAY SNAPSHOT ---')
sal_by_date = df_job_snapshots.groupby('snapshot_date')[['salary_min_vnd', 'salary_max_vnd']].mean().map(lambda x: f'{x:,.0f} đ' if pd.notna(x) else '')
print(sal_by_date.to_string())

print('\n--- 4. PREVIEW 10 DONG BANG job_snapshots (CHUAN POSTGRESQL) ---')
preview_cols = ['snapshot_id', 'job_id', 'snapshot_date', 'salary_min_vnd', 'salary_max_vnd', 'is_active']
print(df_job_snapshots[preview_cols].head(10).to_string(index=False))

--- 1. CAU TRUC BANG job_snapshots (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 3837 entries, 0 to 3836
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   snapshot_id     3837 non-null   int64  
 1   job_id          3837 non-null   int64  
 2   snapshot_date   3837 non-null   str    
 3   salary_min_vnd  1326 non-null   float64
 4   salary_max_vnd  1593 non-null   float64
 5   is_active       3837 non-null   bool   
 6   created_at      3837 non-null   str    
dtypes: bool(1), float64(2), int64(2), str(2)
memory usage: 318.6 KB
None

--- 2. PHAN BO BAN GHI SNAPSHOT THEO NGAY ---
               So snapshot  Ty le (%)
snapshot_date                        
2026-10-11            1510      39.35
2026-10-12            2327      60.65

--- 3. THONG KE MUC LUONG TRUNG BINH THEO TUNG NGAY SNAPSHOT ---
              salary_min_vnd salary_max_vnd
snapshot_date                              
2026-10-11      22,771,808 